# Contacts × Grants Matcher — v4
Rewritten to use `src.modules`. Notebooks cells are now pure orchestration: configure → load → match → analyze → export.

## 1. Environment setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q openai anthropic tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 17.2 MB/s eta 0:00:00


In [3]:
%cd /content/drive/Shareddrives/Matcher/notebooks/pipelines/matching-contacts-grants

/content/drive/Shareddrives/Matcher/notebooks/pipelines/matching-contacts-grants


## 2. Imports

In [4]:
import os
import re
import glob
import uuid
import time
import random
from datetime import datetime, timedelta

import pandas as pd
import numpy as np

from openai import OpenAI, AsyncOpenAI
from anthropic import Anthropic, AsyncAnthropic, InternalServerError

# --- project modules ---
from src.modules.matcher        import load_grants, get_matches
from src.modules.utils          import load_contacts, extract_domain, clean_dataframe_for_excel
from src.modules.ai_analyzer    import analyze_matches_async, analyze_matches_openai_async
from src.modules.email_generator import generate_subject_line, josiah_copy, generate_body

## 3. Clients & paths

In [5]:
keys_path = '/content/drive/Shareddrives/Matcher/keys'

with open(f'{keys_path}/anthropic.txt') as f:
    _ANTH_KEY = f.read().strip()
with open(f'{keys_path}/openai_key.txt') as f:
    _OAI_KEY = f.read().strip()

anth_client       = Anthropic(api_key=_ANTH_KEY)
anth_client_async = AsyncAnthropic(api_key=_ANTH_KEY)
openai_client     = OpenAI(api_key=_OAI_KEY)
openai_client_async = AsyncOpenAI(api_key=_OAI_KEY)

today = datetime.now().strftime('%Y-%m-%d')

## 4. Run config
Everything you'd normally change between runs lives here.

In [6]:
# ── Save / output ──────────────────────────────────────────────────────────
FILE_SAVE_NAME    = 'allContacts'
STARTING_THRESHOLD = 0.84

# ── Email copy ─────────────────────────────────────────────────────────────
CUSTOM_EMAIL    = True    # generate AI subject + body
CUSTOM_FRACTION = 1.0     # fraction of matches to generate copy for (1.0 = all)

# ── Contact filters ────────────────────────────────────────────────────────
CONTACTS_EXCLUDED = ['vc', 'funded', 'linkedin-contacts']
CONTACTS_INCLUDED = ['apollo','sba'] # set to e.g. 'apollo' to restrict to one source

# ── Previous-match dedup window ────────────────────────────────────────────
DEDUP_DAYS = 1   # skip contacts matched in the last N days

# ── Model selection ────────────────────────────────────────────────────────
ANALYZER_MODEL = 'claude-3-haiku-20240307'
COPY_MODEL     = 'claude-sonnet-4-20250514'

## 5. Grant configuration
Set `status: True` for every agency you want to run this session.
`priority` controls processing order (lower = first).

In [7]:
grants = {
    'DOT':              {'status': False, 'priority': 1},
    'MTEC':             {'status': False, 'priority': 2},
    'CUSTOM':           {'status': True, 'priority': 2},
    'EU-GRANTS':        {'status': False, 'priority': 2},
    'ED':               {'status': False, 'priority': 8},
    'EIC':              {'status': False, 'priority': 8},
    'DOC':              {'status': False, 'priority': 1},
    'GRANTS-GOV':       {'status': False, 'priority': 1},
    'DOE':              {'status': False, 'priority': 6},
    'NAVAIR':           {'status': False, 'priority': 6},
    'SERDP':            {'status': False, 'priority': 3},
    'NOAA':             {'status': False, 'priority': 5},
    'CPRIT':            {'status': False, 'priority': 5},
    'DHS':              {'status': False, 'priority': 4},
    'AFOSR':            {'status': False, 'priority': 4},
    'NCI-Special':      {'status': False, 'priority': 1},
    'HHS':              {'status': False,  'priority': 1},
    'NSF':              {'status': False, 'priority': 1},
    'NASA':             {'status': False, 'priority': 4},
    'ARPA':             {'status': False,  'priority': 5},
    'USDA':             {'status': False, 'priority': 1},
    'EPA':              {'status': False, 'priority': 1},
    'DOD-strat-docs':   {'status': False, 'priority': 4},
    'DOD':              {'status': False,  'priority': 2},
    'CDMRP':            {'status': False, 'priority': 4},
    'ALZDiscovery':     {'status': False, 'priority': 4},
    'SOCOM':            {'status': False, 'priority': 4},
    'DHA':              {'status': False, 'priority': 3},
    'BILLANDMELINDAGATES': {'status': False, 'priority': 4},
    'XTECH':            {'status': False, 'priority': 4},
    'BARDA':            {'status': False, 'priority': 4},
    'DARPA':            {'status': False, 'priority': 1},
    'NIMH':             {'status': False, 'priority': 5},
    'NCCIH':            {'status': False, 'priority': 5},
    'NHLBI':            {'status': False, 'priority': 4},
    'NIDA':             {'status': False, 'priority': 3},
    'NIAMS':            {'status': False, 'priority': 4},
    'NINDS':            {'status': False, 'priority': 4},
    'NAVY':             {'status': False, 'priority': 4},
    'USAF':             {'status': False, 'priority': 4},
    'ARMY':             {'status': False, 'priority': 4},
    'PCORI':            {'status': False, 'priority': 4},
    'NICHD':            {'status': False, 'priority': 4},
    'NIH':              {'status': False, 'priority': 4},
    'SPACEWERX':        {'status': False, 'priority': 4},
    'PAST_WINNERS':     {'status': False, 'priority': 4},
    'DOD_BAA':          {'status': False, 'priority': 4},
    'KRONOS':           {'status': False, 'priority': 4},
    'IJA':              {'status': False, 'priority': 4},
    'CIRM':             {'status': False, 'priority': 4},
    'DLA':              {'status': False, 'priority': 4},
    'ERDC':             {'status': False, 'priority': 4},
    'HHS(OLD)':         {'status': False, 'priority': 1},
    'REQUESTS':         {'status': False, 'priority': 1},
    'FBI':              {'status': False, 'priority': 3},
    'CSOC':             {'status': False, 'priority': 1},
    'DEVCOM':           {'status': False, 'priority': 1},

}

# build priority-sorted run order
grants_priority = (
    pd.DataFrame([
        {'grant': k, 'priority': v['priority']}
        for k, v in grants.items() if v['status']
    ])
    .sort_values(['priority', 'grant'])
    .reset_index(drop=True)
)
grants_priority

,grant,priority
0,CUSTOM,2


## 6. Load grant topics

In [8]:
load_grants(grants, data_path='../data')

Loading: CUSTOM
  159 topics loaded


/content/drive/Shareddrives/Matcher/notebooks/pipelines/matching-contacts-grants/src/modules/matcher.py:62: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  topics = pd.concat(topics_list) if len(topics_list) > 1 else topics_list[0]


### 6a. Optional: filter topics by date / close_date
Uncomment and adjust as needed for the current run.

In [9]:
# ── DOD: date-range filter + close_date lookahead ──────────────────────────
# grants['DOD']['topics'] = (
#     grants['DOD']['topics']
#     .query('scraped_at >= "2026-02-03" and scraped_at <= "2026-02-05"')
#     .reset_index(drop=True)
#     .drop(columns=['topic_number'], errors='ignore')
# )
# mask = (pd.to_datetime(today) - grants['DOD']['topics']['close_date']).dt.days <= -40
# grants['DOD']['topics'] = grants['DOD']['topics'][mask].reset_index(drop=True)

# ── HHS: specific close_date ───────────────────────────────────────────────
# grants['HHS']['topics'] = (
#     grants['HHS']['topics']
#     .query('close_date == "2026-09-05"')
#     .reset_index(drop=True)
# )

# ── specific scrape date ─────────────────────────────────────────────
#grants['USDA']['topics'] = (
#     grants['USDA']['topics']
#     .query('scraped_at == "02-23-2026"')
#     .reset_index(drop=True)
# )

grants['HHS']['topics'] = (
     grants['HHS']['topics']
     .query('scraped_at == "02-24-2026"')
     .reset_index(drop=True)
 )

KeyError: 'topics'

## 7. Load contacts

In [9]:
contacts = load_contacts(
    data_path='../data',
    excluded=CONTACTS_EXCLUDED,
    included=CONTACTS_INCLUDED,
)
n_contacts_before = len(contacts)
print(f'Contacts loaded: {n_contacts_before:,}')

Contacts loaded: 213,121


## 8. Filter out previously matched contacts
Reads all match files from the last `DEDUP_DAYS` days and removes those domains from the contact pool.

In [10]:
cutoff = datetime.now() - timedelta(days=DEDUP_DAYS)

all_match_files = glob.glob('../../all-matches/*') + glob.glob('matches/**/*', recursive=True)

recent_match_files = [
    f for f in all_match_files
    if (m := re.search(r'(\d{4}-\d{2}-\d{2})', f))
    and datetime.strptime(m.group(1), '%Y-%m-%d') >= cutoff
]

print(f'{len(recent_match_files)} match files found in last {DEDUP_DAYS} days')

WEBSITE_COLS = ['companyWebsite', 'Website URL', 'company_url', 'url', 'company_website', 'website_url']

prev_websites = []
for path in recent_match_files:
    try:
        df = pd.read_excel(path) if path.endswith('.xlsx') else pd.read_parquet(path)
        df.columns = [str(c) for c in df.columns]
        col = next((c for c in WEBSITE_COLS if c in df.columns), None)
        if col:
            prev_websites.extend(df[col].dropna().astype(str).tolist())
    except Exception as e:
        print(f'Skipping {path}: {e}')

# normalise to bare domains for comparison
def _bare_domain(url):
    d = extract_domain(url)
    return re.sub(r'^www\.', '', d)

prev_domains = set(map(_bare_domain, prev_websites))

contacts['_domain'] = contacts['companyWebsite'].apply(lambda x: _bare_domain(str(x)) if pd.notna(x) else '')
contacts = contacts[~contacts['_domain'].isin(prev_domains)].drop(columns='_domain').reset_index(drop=True)

n_contacts_after = len(contacts)
print(f'Contacts before dedup: {n_contacts_before:,}')
print(f'Contacts after dedup:  {n_contacts_after:,}  ({n_contacts_before - n_contacts_after:,} removed)')

0 match files found in last 1 days
Contacts before dedup: 213,121
Contacts after dedup:  213,121  (0 removed)


## 9. Pre-run summary

In [11]:
n_grant_rows = sum(len(grants[k]['topics']) for k in grants if grants[k]['status'])

print(f'Agencies active:      {len(grants_priority)}')
print(f'Grant topic rows:     {n_grant_rows:,}')
print(f'Contact rows:         {len(contacts):,}')
print(f'Total comparisons:    {n_grant_rows * len(contacts):,.0f}')
print(f'Threshold:            {STARTING_THRESHOLD}')

Agencies active:      1
Grant topic rows:     159
Contact rows:         213,121
Total comparisons:    33,886,239
Threshold:            0.84


## 10. Main matching loop — Tier 1 (Anthropic + AI copy)

In [ ]:
#!mkdir -p matches/matches/tier1/{today}
#!mkdir -p matches/matches/ai_generated/{today}

FILTER_COLS = ['embeddings_x', 'embeddings_y', 'good_match', 'Unnamed: 29']

for _, grant_row in grants_priority.iterrows():
    agency = grant_row['grant']
    threshold = STARTING_THRESHOLD
    n_saved = 0

    print(f'\n=== {agency} | threshold={threshold} ===')

    # ── Step 1: cosine similarity ──────────────────────────────────────────
    matches = get_matches(threshold, grants[agency]['topics'], contacts)
    print(f'  Vector matches: {len(matches)}')

    if len(matches) == 0:
        print(f'  No matches above threshold — skipping.')
        continue

    # ── Step 2: LLM yes/no filter ──────────────────────────────────────────
    matches = await analyze_matches_async(
        matches,
        anth_client_async,
        model=ANALYZER_MODEL,
    )

    matches = (
        matches
        .fillna('-')
        .query('good_match.str.contains("yes", case=False)')
        .drop_duplicates('companyWebsite')
        .reset_index(drop=True)
    )
    print(f'  After LLM filter: {len(matches)}')

    if len(matches) == 0:
        print(f'  No good matches — skipping.')
        continue


        # ── Collect 'no' rows for Tier 2 pivot analysis ────────────────────────
    nos = (
        matches
        .fillna('-')
        .query('~good_match.str.contains("yes", case=False)')
        .drop_duplicates('companyWebsite')
        .reset_index(drop=True)
    )
    if 'tier1_nos' not in dir():
        tier1_nos = nos
    else:
        tier1_nos = pd.concat([tier1_nos, nos]).drop_duplicates('companyWebsite').reset_index(drop=True)


    # ── Step 3: strip embedding/score columns for export ──────────────────
    export_cols = [
        c for c in matches.columns
        if 'embedding' not in c and 'score' not in c and 'good_match' not in c
    ]
    matches_export = matches[export_cols].copy()

    # ── Step 4: AI copy (subject + josiah hook) ────────────────────────────
    if CUSTOM_EMAIL and len(matches_export) >= 1:
        sample_idx = matches_export.sample(frac=CUSTOM_FRACTION).index
        custom = matches_export.loc[sample_idx].copy()
        custom['subject_line'] = None
        custom['ai_message']   = None

        for idx, row in custom.iterrows():
            custom.at[idx, 'subject_line'] = generate_subject_line(
                row['company_summary'], row['agency'], openai_client, anth_client
            )

            attempt = 0
            while True:
                try:
                    custom.at[idx, 'ai_message'] = josiah_copy(
                        row['company_summary'], row['grant_summary'],
                        openai_client, anth_client, model=COPY_MODEL
                    )
                    break
                except InternalServerError as e:
                    attempt += 1
                    wait = min(600, (2 ** attempt) + random.random())
                    print(f'  [{idx}] josiah_copy error, retrying in {wait:.1f}s...')
                    time.sleep(wait)
                except Exception as e:
                    print(f'  [{idx}] josiah_copy unexpected error: {e}')
                    custom.at[idx, 'ai_message'] = ''
                    break

            print(f'  [{idx}] {custom.at[idx, "subject_line"]}')

        custom = (
            clean_dataframe_for_excel(custom)
            .assign(subject_line=lambda x: x['subject_line'].str.replace('"', '', regex=False))
            .assign(ai_message=lambda x: x['ai_message'].str.replace('"', '', regex=False))
        )

        fname = f'matches/matches/ai_generated/{FILE_SAVE_NAME}_{agency}_{uuid.uuid1().hex[:5]}_customSubject_{today}.xlsx'
        custom.to_excel(fname, index=False)
        print(f'  Saved AI copy → {fname}')

        # remaining rows (if CUSTOM_FRACTION < 1.0) go to standard tier1
        matches_export = matches_export.loc[~matches_export.index.isin(sample_idx)]

    # ── Step 5: standard tier1 export ─────────────────────────────────────
    if len(matches_export) > 0:
        fname = f'matches/matches/tier1/{today}/{FILE_SAVE_NAME}_{agency}_{uuid.uuid1().hex[:5]}_{today}.xlsx'
        clean_dataframe_for_excel(matches_export).to_excel(fname, index=False)
        n_saved = len(matches_export)
        print(f'  Saved tier1 → {fname}')

    # ── Step 6: remove matched contacts from pool ──────────────────────────
    matched_urls = matches.companyWebsite.unique()
    contacts = contacts[~contacts['companyWebsite'].isin(matched_urls)].reset_index(drop=True)

    print(f'  Saved: {n_saved} | Contacts remaining: {len(contacts):,}')

## 11. Optional: Tier 2 loop (Pivot Analysis)
Run this against whatever contacts are left after the Tier 1 loop.

In [ ]:
# ── Tier 2: pivot analysis on "no" rows ────────────────────────────────────
# Runs after the Tier 1 loop. Takes every row that got a "no" in the primary
# LLM filter and asks whether the tech could be adapted to meet the grant.
# Passing rows get a pivot_note, then are exported to matches/matches/tier2/.

from src.modules.ai_analyzer import analyze_pivot_async, generate_pivot_notes

!mkdir -p matches/matches/tier2/{today}

# all_nos is accumulated during the Tier 1 loop above — see note below
for _, grant_row in grants_priority.iterrows():
    agency = grant_row['grant']

    print(f'\n=== Pivot check: {agency} ===')

    # re-run cosine matching at the same threshold against the *current* contacts pool
    # (contacts that already got a "yes" in Tier 1 have been removed from the pool,
    #  so this only touches unmatched contacts)
    matches_all = get_matches(STARTING_THRESHOLD, grants[agency]['topics'], contacts)

    if len(matches_all) == 0:
        print('  No candidates — skipping.')
        continue

    # primary yes/no pass (same as Tier 1)
    matches_all = await analyze_matches_async(
        matches_all,
        anth_client_async,
        model=ANALYZER_MODEL,
    )

    # keep only the "no" rows for pivot re-evaluation
    no_matches = (
        matches_all
        .fillna('-')
        .query('~good_match.str.contains("yes", case=False)')
        .drop_duplicates('companyWebsite')
        .reset_index(drop=True)
    )
    print(f'  "No" rows for pivot check: {len(no_matches)}')

    if len(no_matches) == 0:
        continue

    # pivot yes/no pass
    no_matches = await analyze_pivot_async(
        no_matches,
        anth_client_async,
        model=ANALYZER_MODEL,
    )

    pivot_matches = (
        no_matches
        .query('pivot_possible.str.contains("yes", case=False)')
        .reset_index(drop=True)
    )
    print(f'  Pivot-possible rows: {len(pivot_matches)}')

    if len(pivot_matches) == 0:
        continue

    # generate pivot notes for passing rows
    pivot_matches = await generate_pivot_notes(
        pivot_matches,
        anth_client_async,
        model=COPY_MODEL,
    )

    # strip embedding/score columns
    export_cols = [
        c for c in pivot_matches.columns
        if 'embedding' not in c and 'score' not in c
    ]

    fname = f'matches/matches/tier2/{today}/{FILE_SAVE_NAME}_{agency}_{uuid.uuid1().hex[:5]}_pivot_{today}.xlsx'
    clean_dataframe_for_excel(pivot_matches[export_cols]).to_excel(fname, index=False)
    print(f'  Saved pivot matches → {fname}  ({len(pivot_matches)} rows)')

## 12. Export unmatched contacts

In [ ]:
# Export remaining (unmatched) contacts
#unmatched_path = f'matches/matches/unmatched/{FILE_SAVE_NAME}_{uuid.uuid1().hex[:5]}_{today}.xlsx'
#!mkdir -p matches/matches/unmatched
#clean_dataframe_for_excel(contacts).to_excel(unmatched_path, index=False)
#print(f'Unmatched contacts saved → {unmatched_path}  ({len(contacts):,} rows)')

Unmatched contacts saved → matches/matches/unmatched/allContacts_f274c_2026-03-11.xlsx  (213,098 rows)


---
## Scratch / ad-hoc testing
Cells below are for one-off tests — not part of the main pipeline.

In [ ]:
# Quick test: generate_body on a specific company×grant pair
#
# company_summary = """..."""
# grant_summary   = """..."""
# generate_body(company_summary, grant_summary, 'NIDDK', anth_client)

In [ ]:
# Quick test: josiah_copy
#
# josiah_copy(company_summary, grant_summary, anth_client)